In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(api_key = os.getenv("OPENAI_API_KEY"),
                  model="gpt-5-mini", temperature=0.7)

In [4]:
email_voo = """
Olá, Maria! Sua reserva no voo G3-1234 da GOL foi confirmada.
Detalhes do seu voo:
Passageiro: Maria Silva
De: São Paulo (GRU)
Para: Salvador (SSA)
Data: 25/11/2025
Embarque: 10:30
Portão de embarque: B-45
Agradecemos a sua preferência.
"""

### 1. Definir o esquema de saída com Pydantic

In [5]:
class DetalhesVoo(BaseModel):
    """
    Informações de uma reserva de voo.
    """
    passageiro: str = Field(..., description="Nome completo do passageiro.")
    numero_voo: str = Field(..., description="O número do voo (ex: G3-1234).")
    origem: str = Field(..., description="Cidade de origem do voo.")
    destino: str = Field(..., description="Cidade de destino do voo.")
    data_voo: str = Field(..., description="Data do voo no formato DD/MM/AAAA.")
    
parser_voo = JsonOutputParser(pydantic_object=DetalhesVoo)

### 2. Criar o Template do Prompt

In [7]:
prompt_voo = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente especialista em extrair detalhes de reservas de voo de um texto. A saída deve ser um objeto JSON."),
    ("human", "Extraia as informações do texto a seguir com base no esquema de saída:\n{schema}\n\nTexto: {email_voo}"),
]).partial(schema=parser_voo.get_format_instructions())

### 3. Invocar o LLM e analisar a resposta


In [8]:
resposta_voo = chat.invoke(prompt_voo.format_messages(email_voo=email_voo))
resultado_voo = parser_voo.parse(resposta_voo.content)

### 4. Acessando os Dados

In [15]:
resultado_voo

{'passageiro': 'Maria Silva',
 'numero_voo': 'G3-1234',
 'origem': 'São Paulo (GRU)',
 'destino': 'Salvador (SSA)',
 'data_voo': '25/11/2025'}

In [14]:
print("--- Detalhes da Reserva ---")
print(f"Passageiro: {resultado_voo['passageiro']}")
print(f"Número do Voo: {resultado_voo['numero_voo']}")
print(f"Origem: {resultado_voo['origem']}")
print(f"Destino: {resultado_voo['destino']}")
print(f"Data do Voo: {resultado_voo['data_voo']}")

--- Detalhes da Reserva ---
Passageiro: Maria Silva
Número do Voo: G3-1234
Origem: São Paulo (GRU)
Destino: Salvador (SSA)
Data do Voo: 25/11/2025
